# 07b Raw ADC – Load & Plot

Loads `ds_raw` from a local `.nc` file and plots raw ADC traces. Run notebook 01 first to execute and save.

## Imports

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from iqcc_calibration_tools.qualibrate_config.qualibrate.node import QualibrationNode
from quam_builder.architecture.superconducting.qpu import FluxTunableQuam as Quam
from calibration_utils.iq_blobs_raw_adc import Parameters, plot_raw_adc_traces
from qualibration_libs.parameters import get_qubits

## Node initialisation

In [ ]:
description = """
RAW ADC
Collects raw ADC traces. No demodulation or post-processing.
"""

node = QualibrationNode[Parameters, Quam](
    name="07b_iq_blobs_raw_adc",
    description=description,
    parameters=Parameters(),
)

@node.run_action(skip_if=node.modes.external)
def custom_param(node: QualibrationNode[Parameters, Quam]):
    node.parameters.qubits = ["qB5"]
    pass

node.machine = Quam.load()

# Local path for loading ds_raw (same as in notebook 01)
DS_RAW_PATH = os.path.join(os.getcwd(), "07b_iq_blobs_raw_adc_ds_raw.nc")

## Load saved data

In [ ]:
if os.path.exists(DS_RAW_PATH):
    dataset = xr.open_dataset(DS_RAW_PATH)
    node.results["ds_raw"] = dataset
    node.namespace["qubits"] = get_qubits(node)
    print(f"Loaded ds_raw from {DS_RAW_PATH}")
else:
    print(f"File not found: {DS_RAW_PATH}. Run notebook 01 first to execute and save.")

## Plot raw ADC traces

In [ ]:
if "ds_raw" in node.results:
    fig = plot_raw_adc_traces(node.results["ds_raw"], node.namespace["qubits"])
    node.add_node_info_subtitle(fig)
    plt.show()